# new_table

Makes a Table folder for each new Saved table: from its query, when QUERIES gives one, its logic written for you; else from a Template, its logic for you to write. A Building block, a piece several notebooks share, is made with project.new_block(...).

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Table folder**: the folder in tables/ that holds everything about one Saved table: its Table notebook, its past lineage, and its Hive.
- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Template**: the empty shape of a Table notebook, in admin/templates/, each specific left as a `<...>` to fill in.
- **Building block**: a piece of your own code several notebooks share, a file of its own in building_blocks/.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Batch**: the scripts converted together, under one name.
- **Table notebook**: the notebook in a Table folder whose cells create, preview and write its Saved table; its logic cell is the one whose first line is `# logic`.
- **Key**: the columns whose values name exactly one row of a table.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Dev Database**: the database where you create and write tables of your own, each built, written and compared there first: `DEV` in admin/databases.py.
- **Hive**: the SQL your warehouse runs.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.

## Before step 1

Fill in the names cell below and save the notebook: each call reads it, and so do `project.check()`, which names the next step, and `project.do_next()`, the cell before the last.

## Decision rules

- If you run the steps yourself, run each step's cell in turn, in order.
- If you want them run for you, run the last cell, `project.run_workflow(...)`, in place of their cells. It runs every step left in turn, up to the first only you can do, and says where it stopped: go on from that step.
- If you want only the steps that change files run for you, run `project.do_next()`, the cell before the last. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each table in `TABLES` has its Table folder in tables/, and its step 2 cell runs with no error.
- Each table's Table reference lists its columns and leaves no Decision open.
- admin/sent.csv records each table created and written.
- Each table made from a query has a `conversion/compare.md` that says it passed.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
TABLES = ["mart_dev.failed_runs"]  # the new tables, by their names in the warehouse, such as ["mart_dev.daily_job_runs"]
QUERIES = {"failed_runs": "failed_runs.hql"}  # each table's query, by its name without its database, as Hive text or a .hql file in admin/workflows/new_table/, such as {"daily_job_runs": "daily_job_runs.hql"} for mart_dev.daily_job_runs: it is made in your Dev Database, its logic written from its query

In [ ]:
DECISIONS = {"mart_dev.failed_runs": {"location": "/data/team/mart_dev.db/failed_runs"}}  # your answer to each open Decision, by the table's name


## 1. Make each Table folder, converting each table's query QUERIES gives

It changes only files, so project.do_next() runs it for you.

- **Goal:** a Table folder for each table in `TABLES`: from its query, when `QUERIES` gives one, its logic written for you; else from a Template, its logic for you to write.
- **Action:** If `QUERIES` gives a query, first give each table it reads a Table reference, with the references Workflow. Then run the cell below. Each query is converted as the convert Workflow converts one: in a batch of its own, named for its table, in `admin/workflows/convert/`; the batch moves to `finished/` once compared.
- **Done when** each table in `TABLES` has its Table folder in tables/; then step 2.

In [ ]:
project.new_table(TABLES, queries=QUERIES)

## 2. Write each table's logic in its Table notebook, and save it

It is a decision, yours to make.

- **Goal:** logic in each Table notebook that loads, so its `preview()` builds.
- **Action:** Open each table's Table notebook, in its Table folder, fill in each `<...>` in its code cells as its text cell says, and save it; a table made from its query has its logic written already. Then run the cell below: it says what stops each table's logic, if anything does.
- **Done when** the cell below runs with no error; then step 3.

In [ ]:
for table in TABLES:
    project.load(table).preview()  # says what stops it, if anything does

## 3. Write each Table reference from its logic

It changes only files, so project.do_next() runs it for you.

- **Goal:** each table's Table reference written from its logic: its columns and types from `preview()`, its partition columns from `PARTITIONED_BY`, and the Key its query makes.
- **Action:** Run the cell below.
- **Done when** each table's Table reference in references/ lists its columns, no longer the stand-in new_table wrote; then step 4.

In [ ]:
project.reference_from_logic(TABLES)

## 4. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each open Decision of the new tables.
- **Action:** Write each answer in the decisions cell, `DECISIONS`, by the table's name as `project.check()` writes it, then save the notebook.
- **Done when** the decisions cell answers each Decision `project.check()` names for these tables; then step 5.

## 5. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its table's Table reference.
- **Action:** Run the cell below.
- **Done when** each table's Table reference leaves no Decision open; then step 6.

In [ ]:
project.write_decisions(DECISIONS)

## 6. Check each table's rows with preview(), which writes nothing

It is a call to the warehouse, yours to run.

- **Goal:** each table's rows read and checked before anything is written.
- **Action:** Run the cell below. It prints the rows each table's `preview()` gives, read through your query. `project.check()` names this step beside step 7.
- **Done when** you have read each table's rows, and they are the rows it should hold; then step 7.

In [ ]:
for table in TABLES:
    print(run_query(project.load(table).preview(), query))

## 7. Create and write each in dev

It is a call to the warehouse, yours to run.

- **Goal:** each table made and written in your Dev Database.
- **Action:** Run the cell below. It sends each table's `create()`, then `project.run_tables` sends each table's `write()`, through your execute.
- **Done when** admin/sent.csv records each table written; then step 8.

In [ ]:
for table in TABLES:
    run_execute(project.load(table).create(), execute)
project.run_tables(tables=TABLES, execute=execute)

## 8. Compare each table made from a query with its query

It is a call to the warehouse, yours to run.

- **Goal:** each table made from a query checked against its query.
- **Action:** Run the cell below. `project.compare_batch` compares each with its query, through your query, and writes the result in its `conversion/compare.md`. If `QUERIES` gives no query, this step is done once step 7 is.
- **Done when** each table made from a query has a `conversion/compare.md` that says it passed; the Workflow is done: go through the Self-check at the top.

In [ ]:
for name in QUERIES:
    print(project.compare_batch(name, query=query))

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("new_table", query=query, execute=execute)